# MetaRCWA Square Particle

This notebook constructs and solves one periodic square particle structure using Metarcwa.


In [2]:
import torch

from dataclasses import dataclass
from pathlib import Path

import yaml

In [3]:
@dataclass
class SquareParticleModel:
    """Physical description of one periodic square particle structure.

    All lengths are expressed in nanometers. Angles are expressed in degrees because
    that is what S4 expects. 

    Attributes
    ----------
    period_nm: 
        Period of the square lattice in both x and y
    particle_side_nm:
        Full side length of the square particle.
    planarization_thickness_nm:
        Thickness of the homogeneous planarization layer.
    patterned_layer_thickness_nm:
        Thickness of the layer containing the square particle. 
    wavelength_nm:
        Free-space wavelength.
    theta_deg:
        Polar incidence angle measured from the surface normal.
    phi_deg:
        Azimuthal incidence angle.
    incidence_epsilon:
        Permittivity of the semi-infinite incidence material
    planarization_epsilon:
        Permittivity of the homogeneous planarization layer
    background_epsilon:
        Permittivity of the material surrounding the square particle
    particle_epsilon:
        Permittivity of the square particle.
    transsmission_epsilon:
        Permittivity of the semi-infinite transmission material
    """

    period_nm: float
    particle_side_nm: float
    planarization_thickness_nm: float
    patterned_layer_thickness_nm: float
    wavelength_nm: float
    theta_deg: float
    phi_deg: float
    incidence_epsilon: complex
    planarization_epsilon: complex
    background_epsilon: complex
    particle_epsilon: complex
    transmission_epsilon: complex

    @classmethod
    def from_dict(cls, data:dict) -> "SquareParticleModel":
        """Create a physical model from YAML-derived dictionary data."""

        # Copy the dictionary so we do not modify the original data
        data = dict(data)

        epsilon_list = (
            "incidence_epsilon",
            "planarization_epsilon",
            "background_epsilon", 
            "particle_epsilon",
            "transmission_epsilon",
        )

        # Convert each {real:..., image:...} dictionary into 
        # an ordinary Python complex number
        for epsilon in epsilon_list:
            components = data[epsilon]

            data[epsilon] = complex(
                components["real"],
                components.get("imag",0.0)
            )

        return cls(**data)

In [4]:
@dataclass
class S4Config:
    """Numerical settings used directly by S4

    Attributes
    ----------
    num_basis:
        Maximum number of Fourier orders to retain. Due to truncation, 
        the actual number may be different.
    lattice_truncation:
        While num_basis gives how many fourier orders to approximately retain,
        lattice truncation says how to choose them. 
        In S4, this can be 'Circular' or 'Parallelogramic'.
    """

    num_basis: int
    lattice_truncation: str
    DiscretizedEpsilon: bool
    DiscretizationResolution: int

    def __post_init__(self) -> None:
        """Check that the supplied settings are valid."""

        if self.num_basis <=0:
            raise ValueError("num_basis must be positive")

        allowed_truncations = {
            "Circular",
            "Parallelogramic"
        }

        if self.lattice_truncation not in allowed_truncations:
            raise ValueError(
                "lattice_truncation must be "
                "'Circular' or 'Parallelogramic'"
            )

    @classmethod
    def from_dict(cls, data:dict) -> "S4Config":
        "Create the numerical settings from YAML derived dictionary data."

        # Copy the dictionary so you don't modify the original data
        data = dict(data)

        return cls(**data)

In [5]:
config_path = Path("../src/configs/square_particle.yaml")

with config_path.open("r") as file:
    yaml_data = yaml.safe_load(file)

In [6]:
model = SquareParticleModel.from_dict(
    yaml_data["model"]
)

config = S4Config.from_dict(
    yaml_data["s4"]
)